## 1. Постановка задачи

К вам обратился представитель крупного агентства недвижимости со
следующей проблемой:

"Мои риелторы тратят катастрофически много времени на сортировку
объявлений и поиск выгодных предложений. Поэтому их скорость реакции, да
и, сказать по правде, качество анализа не дотягивают до уровня конкурентов.
Это сказывается на наших финансовых показателях."

**Техническая задача:** разработать модель, которая позволила бы обойти
конкурентов по скорости и качеству совершения сделок.

## 2. Знакомство с данными, базовый анализ

In [7]:
import ast

import pandas as pd

housing_data = pd.read_csv('housing_data.csv')
housing_data.head()

,status,private pool,propertyType,street,baths,homeFacts,fireplace,city,schools,sqft,zipcode,beds,state,stories,mls-id,PrivatePool,MlsId,target
0,Active,NaN,Single Family Home,240 Heather Ln,3.5,"{'atAGlanceFacts': [{'factValue': '2019', 'fac...",Gas Logs,Southern Pines,"[{'rating': ['4', '4', '7', 'NR', '4', '7', 'N...",2900,28387,4,NC,NaN,NaN,NaN,611019,"$418,000"
1,for sale,NaN,single-family home,12911 E Heroy Ave,3 Baths,"{'atAGlanceFacts': [{'factValue': '2019', 'fac...",NaN,Spokane Valley,"[{'rating': ['4/10', 'None/10', '4/10'], 'data...","1,947 sqft",99216,3 Beds,WA,2.0,NaN,NaN,201916904,"$310,000"
2,for sale,NaN,single-family home,2005 Westridge Rd,2 Baths,"{'atAGlanceFacts': [{'factValue': '1961', 'fac...",yes,Los Angeles,"[{'rating': ['8/10', '4/10', '8/10'], 'data': ...","3,000 sqft",90049,3 Beds,CA,1.0,NaN,yes,FR19221027,"$2,895,000"
3,for sale,NaN,single-family home,4311 Livingston Ave,8 Baths,"{'atAGlanceFacts': [{'factValue': '2006', 'fac...",yes,Dallas,"[{'rating': ['9/10', '9/10', '10/10', '9/10'],...","6,457 sqft",75205,5 Beds,TX,3.0,NaN,NaN,14191809,"$2,395,000"
4,for sale,NaN,lot/land,1524 Kiscoe St,NaN,"{'atAGlanceFacts': [{'factValue': '', 'factLab...",NaN,Palm Bay,"[{'rating': ['4/10', '5/10', '5/10'], 'data': ...",NaN,32908,NaN,FL,NaN,NaN,NaN,861745,"$5,000"


In [8]:
housing_data.shape

(377185, 18)

# 📑 Описание колонок датасета

Размер датасета: 377,185 строк × 18 колонок

Назначение: предсказание стоимости домов (target)

| Колонка       | Описание |
|---------------|----------|
| **status**    | Статус недвижимости (например, продаётся, сдана, в аренде и т.д.) |
| **private pool** | Наличие частного бассейна (обычно "Yes") |
| **propertyType** | Тип недвижимости (например, single-family home, condo, townhouse) |
| **street**    | Адрес или название улицы |
| **baths**     | Количество ванных комнат |
| **homeFacts** | Дополнительные характеристики дома (год постройки, материалы, особенности) |
| **fireplace** | Наличие камина |
| **city**      | Город, в котором находится дом |
| **schools**   | Информация о школах рядом (рейтинг, количество) |
| **sqft**      | Площадь дома в квадратных футах |
| **zipcode**   | Почтовый индекс |
| **beds**      | Количество спален |
| **state**     | Штат (например, CA, TX, NY) |
| **stories**   | Количество этажей в доме |
| **mls-id**    | Идентификатор MLS (Multiple Listing Service, система недвижимости) |
| **PrivatePool** | Дублирующая колонка про наличие бассейна |
| **MlsId**     | Альтернативная версия идентификатора MLS |
| **target**    | Цена недвижимости (целевая переменная для предсказания) |

In [9]:
housing_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 377185 entries, 0 to 377184
Data columns (total 18 columns):
 #   Column        Non-Null Count   Dtype 
---  ------        --------------   ----- 
 0   status        337267 non-null  object
 1   private pool  4181 non-null    object
 2   propertyType  342452 non-null  object
 3   street        377183 non-null  object
 4   baths         270847 non-null  object
 5   homeFacts     377185 non-null  object
 6   fireplace     103114 non-null  object
 7   city          377151 non-null  object
 8   schools       377185 non-null  object
 9   sqft          336608 non-null  object
 10  zipcode       377185 non-null  object
 11  beds          285903 non-null  object
 12  state         377185 non-null  object
 13  stories       226469 non-null  object
 14  mls-id        24942 non-null   object
 15  PrivatePool   40311 non-null   object
 16  MlsId         310305 non-null  object
 17  target        374704 non-null  object
dtypes: object(18)
memory usa

In [10]:
def missing_percent(feature):
    return round(housing_data[feature].isna().sum() / housing_data.shape[0] * 100)


for feature in housing_data.columns:
    percent = missing_percent(feature)
    print("Missing percent for", feature, "is", percent, "%")

Missing percent for status is 11 %
Missing percent for private pool is 99 %
Missing percent for propertyType is 9 %
Missing percent for street is 0 %
Missing percent for baths is 28 %
Missing percent for homeFacts is 0 %
Missing percent for fireplace is 73 %
Missing percent for city is 0 %
Missing percent for schools is 0 %
Missing percent for sqft is 11 %
Missing percent for zipcode is 0 %
Missing percent for beds is 24 %
Missing percent for state is 0 %
Missing percent for stories is 40 %
Missing percent for mls-id is 93 %
Missing percent for PrivatePool is 89 %
Missing percent for MlsId is 18 %
Missing percent for target is 1 %


# 📉 Анализ пропусков в данных

## 🔎 Общие наблюдения
- В датасете присутствует **значительное количество пропусков** в ряде колонок.
- Наиболее критичные признаки с пропусками:
  - `private pool` — 98.89% пропусков (практически неинформативный признак).
  - `mls-id` — 93.39% пропусков.
  - `PrivatePool` — 89.31% пропусков (дублирует `private pool`).
  - `fireplace` — 72.66% пропусков.
- Средний уровень пропусков:
  - `stories` — 39.96%.
  - `baths` — 28.19%.
  - `beds` — 24.20%.
  - `MlsId` — 17.73%.
  - `sqft` — 10.76%.
  - `status` — 10.58%.
  - `propertyType` — 9.21%.
- Почти полное отсутствие пропусков в:
  - `target` (цена) — только 0.66%.
  - `city`, `street` — менее 0.01%.
  - `zipcode`, `schools`, `state`, `homeFacts` — без пропусков.

## ✅ Выводы
1. Колонки `private pool` и `PrivatePool` имеют слишком много пропусков и дублируют друг друга. Однако скорей всего эти значения обозначают, что в доме нет бассейна и их можно заменить на "No". Так же целесообразно оставить только одну колонку.
2. Колонка `fireplace` имеет тоже много пропусков, возможно пропуски тоже обозначают "No".
3. Колонка `mls-id` также содержит слишком много пропусков и может быть исключена.
4. В колонках `stories`, `baths`, `beds`, `sqft` пропуски значительные, но эти признаки важны — стоит рассмотреть методы заполнения (например, медианой, модой или предсказанием модели).
5. Целевая переменная `target` имеет мало пропусков (0.66%) — можно удалить эти строки без ущерба для модели.
6. Основные категориальные признаки (`state`, `zipcode`, `schools`) заполнены полностью — они могут быть полезными для обучения.


In [11]:
housing_data['status'].value_counts()

status
for sale                156104
Active                  105207
For sale                 43465
foreclosure               6426
New construction          5475
                         ...  
Contingent   No Show         1
Coming soon: Oct 24.         1
Coming soon: Oct 21.         1
Coming soon: Nov 14.         1
Coming soon: Dec 23.         1
Name: count, Length: 159, dtype: int64

# 🏷️ Анализ колонки `status`

## 📊 Основные значения и их частоты
- `for sale` — 156,104 записей
- `Active` — 105,207 записей
- `For sale` — 43,465 записей (дубликат по смыслу с разным регистром)
- `NaN` — 39,918 записей (10.6% пропусков)
- `foreclosure` — 6,426 записей
- `New construction` — 5,475 записей
- `Pending` — 4,702 записей
- `Pre-foreclosure` — 2,119 записей
- `Pre-foreclosure / auction` — 1,560 записей
- `P` — 1,488 записей (неоднозначное значение, возможно ошибка в данных)
- `Under Contract Show` — 1,183 записей
- ` / auction` — 936 записей (некорректное/обрезанное значение)
- `Under Contract   Showing` — 793 записей (дубликат с другим форматированием)
- `Active Under Contract` — 718 записей
- `Under Contract` — 690 записей
- `New` — 690 записей
- `Contingent` — 581 записей
- `Price Change` — 563 записей
- `Auction` — 536 записей
- `Foreclosed` — 459 записей

## 🔎 Общее количество уникальных значений
- **159 уникальных статусов**

## ✅ Выводы
1. Колонка содержит **дублирующиеся значения в разном регистре** (`for sale` и `For sale`), их нужно унифицировать (привести к нижнему регистру).
2. Есть **мусорные значения**: `P`, ` / auction`, двойные пробелы в строках. Их нужно очистить или объединить с корректными значениями.
3. Основные категории, которые можно выделить:
   - `for sale` / `active` — активные предложения.
   - `pending` / `under contract` / `contingent` — сделки в процессе.
   - `foreclosure` / `pre-foreclosure` / `auction` / `foreclosed` — проблемные или аукционные продажи.
   - `new` / `new construction` — новостройки.
   - `price change` — изменившие цену.
4. Пропусков достаточно много (**10.6%**), стоит рассмотреть заполнение их как отдельной категории (например, `"unknown"`).
5. Для моделирования целесообразно **сжать количество уникальных значений** с 159 → примерно 5–8 крупных категорий.
6. Все значения которые не имеют смысла или не относятся к продаже лучше удалить

In [12]:
pool1 = housing_data['private pool'].value_counts()
pool2 = housing_data['PrivatePool'].value_counts()
print(f"Private pool1: {pool1}\nPrivate pool2: {pool2}")

Private pool1: private pool
Yes    4181
Name: count, dtype: int64
Private pool2: PrivatePool
yes    28793
Yes    11518
Name: count, dtype: int64


# 🏊 Анализ колонок `private pool` и `PrivatePool`

## 📊 Наблюдения
- **`private pool`**
  - Содержит только одно значение `"Yes"`.
  - Пропусков 98.9%.
  - Фактически бинарный индикатор (есть бассейн или нет).

- **`PrivatePool`**
  - Уникальные значения:
    - `yes` — 28,793 раз.
    - `Yes` — 11,518 раз.
    - `NaN` — 336,874 раз.
  - Всего два текстовых значения, отличающихся только регистром (`yes` и `Yes`).
  - Пропусков 89.3%.

## ✅ Вывод
1. Колонка `PrivatePool` содержит больше данных, чем `private pool`, но значения необходимо **привести к одному регистру** (например, `"Yes"`).
2. Колонка `private pool` избыточна и дублирует смысл `PrivatePool`, только в более ограниченном виде.
3. Целесообразно:
   - Привести значения `PrivatePool` к единообразному виду (`"Yes"`) и заменить пропуски на `"No"`.
   - Объединить эти две колонки в одну.
   - Колонки `private pool` и `PrivatePool` можно удалить после объединения информации.

In [13]:
housing_data['propertyType'].value_counts()

propertyType
single-family home                                             92206
Single Family                                                  62869
Single Family Home                                             31728
condo                                                          25968
lot/land                                                       20552
                                                               ...  
Custom, Elevated, Other                                            1
Contemporary, Farmhouse                                            1
2 Stories, Traditional, Mediterranean, Texas Hill Country          1
1 Story, Contemporary, Traditional, Mediterranean                  1
Bilevel, Converted Dwelling, Loft with Bedrooms, Condo/Unit        1
Name: count, Length: 1280, dtype: int64

# 🏠 Анализ колонки `propertyType`

## 📊 Основные значения и частоты
- `single-family home` — 92,206 записей
- `Single Family` — 62,869 записей
- `Single Family Home` — 31,728 записей
- `condo` — 25,968 записей
- `lot/land` — 20,552 записей
- `Condo` — 16,561 записей
- `townhouse` — 11,464 записей
- `Land` — 10,934 записей
- `multi-family` — 7,900 записей
- `Condo/Townhome/Row Home/Co-Op` — 7,701 записей
- `Townhouse` — 6,936 записей
- `Traditional` — 5,913 записей
- Другие значения встречаются реже.

## 🔎 Общая информация
- Всего уникальных значений: **1280**.
- В колонке есть **разные варианты написания одного и того же типа недвижимости** (`single-family home`, `Single Family`, `Single Family Home`).
- Есть **дубликаты по регистру** (`condo` и `Condo`, `townhouse` и `Townhouse`).
- Есть **общие категории** (`lot/land`, `Land`, `multi-family`, `coop`, `High Rise`, `mobile/manufactured`).
- Пропусков: **9.2% (34,733 записей)**.

## ✅ Выводы
1. Колонка содержит полезную информацию о типе недвижимости, но в текущем виде она **сильно зашумлена из-за множества дублей и разных написаний**.
2. Для улучшения качества данных необходимо:
   - Привести значения к **единому регистру** (например, нижний).
   - Объединить синонимичные категории (например, `"single-family home"`, `"Single Family"`, `"Single Family Home"` → `"single family"`).
   - Сократить количество категорий, выделив **основные группы** (например: *single-family, condo, townhouse, multi-family, land, manufactured/mobile, coop, other*).
3. После нормализации количество уникальных категорий можно сократить с **1280 → примерно 10–15 основных типов**, что сделает признак более полезным для моделирования.
4. Пропуски можно заполнить отдельной категорией `"unknown"`.

In [14]:
housing_data['baths'].value_counts()

baths
2 Baths       52466
3 Baths       35506
2             20452
2.0           16576
4 Baths       14764
              ...  
4.75 Baths        1
5.25 Baths        1
41.0              1
1.8 Baths         1
44.0              1
Name: count, Length: 229, dtype: int64

# 🚿 Анализ колонки `baths`

## 📊 Основные значения и частоты
- Пропуски — **106,338** (28.2%).
- Наиболее частые значения:
  - `2 Baths` — 52,466 записей
  - `3 Baths` — 35,506
  - `2` — 20,452
  - `2.0` — 16,576
  - `4 Baths` — 14,764
  - `3.0` — 10,869
  - `3` — 10,113
  - `Bathrooms: 2` — 9,538
  - `2.5` — 8,113
  - `Bathrooms: 3` — 6,613
  - `1` — 6,583
  - `1.0` — 5,930
  - `5 Baths` — 5,370

## 🔎 Общая информация
- Всего уникальных значений: **229**.
- Значения представлены в **разных форматах**:
  - Число (`2`, `3.0`, `2.5`)
  - Текст с числом (`2 Baths`, `Bathrooms: 2`, `2.5 Baths`)
- Есть некорректные/сомнительные значения: `0` (3,811 записей).
- Большинство значений — от **1 до 5 ванных комнат**.

## ✅ Выводы
1. Колонка `baths` содержит важный числовой признак, но данные сильно "зашумлены" из-за разных форматов записи.
2. Для анализа и моделирования необходимо:
   - Преобразовать все значения к **числовому виду** (float).
   - Удалить текстовые части (`Baths`, `Bathrooms:`).
   - Привести дробные значения (например, `2.5`) к числу **2.5**.
3. Значение `0` скорее всего означает отсутствие ванной комнаты, его можно оставить как есть или рассмотреть как пропуск.
4. После очистки колонка будет числовой и станет полезной для предсказательной модели.

In [15]:
housing_data['homeFacts'].iloc[0]

"{'atAGlanceFacts': [{'factValue': '2019', 'factLabel': 'Year built'}, {'factValue': '', 'factLabel': 'Remodeled year'}, {'factValue': 'Central A/C, Heat Pump', 'factLabel': 'Heating'}, {'factValue': '', 'factLabel': 'Cooling'}, {'factValue': '', 'factLabel': 'Parking'}, {'factValue': None, 'factLabel': 'lotsize'}, {'factValue': '$144', 'factLabel': 'Price/sqft'}]}"

In [16]:
all_fields = set()
for val in housing_data['homeFacts'].values:
    if val:
        data = ast.literal_eval(val)
        facts = data.get('atAGlanceFacts', [])
        for fact in facts:
            if fact.get('factLabel'):
                all_fields.add(fact['factLabel'])
all_fields

{'Cooling',
 'Heating',
 'Parking',
 'Price/sqft',
 'Remodeled year',
 'Year built',
 'lotsize'}

# 🏡 Анализ колонки `homeFacts`

## 📊 Наблюдения
- Колонка содержит **словарь с характеристиками дома** в виде текста (JSON-подобные строки).
- Основные поля внутри:
  - `Year built` — год постройки.
  - `Remodeled year` — год последнего ремонта (может быть пустым).
  - `Heating` — тип отопления (например, Forced Air, Central).
  - `Cooling` — тип охлаждения (например, Central Air).
  - `Parking` — информация о парковке (количество мест, гараж, carport и т.д.).
  - `lotsize` — размер участка (в sqft или acres).
  - `Price/sqft` — стоимость за квадратный фут.

## 🔎 Проблемы
- Данные хранятся в виде текста, поэтому **неструктурированные** и требуют парсинга.
- Форматы значений разные: `7289`, `0.44 acres`, `9,583 sqft lot`, `—`.
- Есть пустые или `None` значения в некоторых полях.
- Некоторые признаки дублируют информацию из других колонок (например, `Price/sqft` связан с `target` и `sqft`).

## ✅ Выводы
1. Колонка `homeFacts` является **составным источником признаков**, и в текущем виде использовать её нельзя — необходимо распарсить JSON-структуру.
2. Полезные признаки для выделения:
   - `year_built` (год постройки).
   - `remodeled_year` (год ремонта).
   - `heating_type`.
   - `cooling_type`.
   - `parking_spaces`.
   - `lot_size` (привести все форматы к sqft).
   - `price_per_sqft` (числовой признак).
3. После выделения этих признаков их можно использовать в модели для повышения точности предсказаний.
4. Исходная колонка после парсинга может быть удалена, чтобы не хранить дублирующую текстовую информацию.

In [17]:
housing_data['fireplace'].value_counts()

fireplace
yes                                                                     50356
Yes                                                                     20856
1                                                                       14544
2                                                                        2432
Not Applicable                                                           1993
                                                                        ...  
Free-standing, Insert, Wood                                                 1
Wood Burning, Attached Fireplace Doors/Screen, Electric, Gas Starter        1
One, Living Room                                                            1
FAMILYRM, Great Room, Living Room                                           1
Ceiling Fan, SMAPL, Utility Connection, Walk-In Closets                     1
Name: count, Length: 1652, dtype: int64

# 🔥 Анализ колонки `fireplace`

## 📊 Основные значения и частоты
- Пропуски — **274,070** (72.7%).
- Наиболее частые значения:
  - `yes` — 50,356
  - `Yes` — 20,856
  - `1` — 14,544
  - `2` — 2,432
  - `Not Applicable` — 1,993
  - `Fireplace` — 847
  - `3` — 564
  - `Living Room` — 433
  - `LOCATION` — 399
  - `Wood Burning` — 311
  - `Gas/Gas Logs` — 300
  - `No` — 289
  - `Fireplace YN` — 287
  - `Special Features` — 279

## 🔎 Общая информация
- Всего уникальных значений: **1,653**.
- Значения сильно "зашумлены" и содержат:
  - Булевый формат (`yes`, `Yes`, `No`).
  - Количественный формат (`1`, `2`, `3`).
  - Описательный формат (`Wood Burning`, `Gas/Gas Logs`, `Living Room`, `Great Room`).
  - Служебные/мусорные строки (`Fireplace`, `Fireplace YN`, `LOCATION`).
- Таким образом, колонка объединяет несколько разных типов информации:
  - факт наличия (`Yes` / `No`),
  - количество каминов (`1`, `2`, `3`),
  - тип камина (wood burning, gas).
- Очень высокая доля пропусков (~73%).

## ✅ Выводы
1. Колонка `fireplace` содержит полезную информацию, но требует серьёзной очистки и нормализации.
2. Рекомендуется разделить информацию на отдельные признаки:
   - **fireplace_present** — бинарный (есть/нет).
   - **fireplace_count** — числовой (количество).
   - **fireplace_type** — категориальный (wood, gas, electric и т.п.).
3. Строки-шум (`Fireplace`, `Fireplace YN`, `LOCATION`) можно удалить или заменить на пропуски.
4. Значения `"Yes"` и `"yes"` привести к одному виду.
5. Несмотря на большое количество пропусков, признак может быть полезен после очистки — наличие камина и его тип могут влиять на цену дома.

In [18]:
housing_data['beds'].value_counts()

beds
3 Beds         53459
4 Beds         35418
3              31406
2 Beds         26362
4              20030
               ...  
8,023 sqft         1
10,193 sqft        1
8.93 acres         1
5,510 sqft         1
8,479 sqft         1
Name: count, Length: 1184, dtype: int64

# 🛏️ Анализ колонки `beds`

## 📊 Основные значения и частоты
- Пропуски — **91,282** (24.2%).
- Наиболее частые значения:
  - `3 Beds` — 53,459
  - `4 Beds` — 35,418
  - `3` — 31,406
  - `2 Beds` — 26,362
  - `4` — 20,030
  - `2` — 16,110
  - `Baths` — 15,282 (ошибочное значение)
  - `3 bd` — 12,877
  - `5 Beds` — 11,271
  - `4 bd` — 8,265
  - `3.0` — 8,088
  - `5` — 6,424
  - `2 bd` — 5,243
  - `4.0` — 5,231
  - `1` — 4,610
  - `6 Beds` — 3,810

## 🔎 Общая информация
- Всего уникальных значений: **1184**.
- Значения представлены в **разных форматах**:
  - Число (`3`, `4.0`, `2.0`).
  - Текст с числом (`3 Beds`, `4 bd`).
  - Ошибочные записи (`Baths`).
- Большинство домов имеют **от 1 до 6 спален**.
- Есть дробные значения (`3.0`, `2.0`), которые дублируют целые.

## ✅ Выводы
1. Колонка `beds` содержит важный числовой признак, но требует **очистки и нормализации**.
2. Необходимо:
   - Удалить текстовые части (`Beds`, `bd`) и привести все значения к числу.
   - Исправить ошибочные значения (`Baths` → NaN).
   - Привести дробные (`3.0`) к целым числам.
3. После преобразования колонка станет **целочисленной** (`int`), что значительно повысит её ценность для модели.
4. Пропуски (24.2%) можно заполнить медианой, либо отдельной категорией `"unknown"`.

In [19]:
housing_data['city'].head()

0    Southern Pines
1    Spokane Valley
2       Los Angeles
3            Dallas
4          Palm Bay
Name: city, dtype: object

In [20]:
ast.literal_eval(housing_data['schools'].iloc[0])

[{'rating': ['4', '4', '7', 'NR', '4', '7', 'NR', 'NR'],
  'data': {'Distance': ['2.7 mi',
    '3.6 mi',
    '5.1 mi',
    '4.0 mi',
    '10.5 mi',
    '12.6 mi',
    '2.7 mi',
    '3.1 mi'],
   'Grades': ['3–5', '6–8', '9–12', 'PK–2', '6–8', '9–12', 'PK–5', 'K–12']},
  'name': ['Southern Pines Elementary School',
   'Southern Middle School',
   'Pinecrest High School',
   'Southern Pines Primary School',
   "Crain's Creek Middle School",
   'Union Pines High School',
   'Episcopal Day Private School',
   'Calvary Christian Private School']}]

# 🏫 Анализ колонки `schools`

## 📊 Наблюдения
- Колонка хранит данные в виде списка словарей (JSON-подобная структура).
- Каждый элемент списка описывает одну школу рядом с домом и содержит:
  - **`name`** — название школы (например, *North Atlanta High School*).
  - **`rating`** — рейтинг школы (форматы: `6`, `6/10`, `NR` — no rating).
  - **`data.Grades`** — диапазон классов (например, `PK–5`, `9–12`).
  - **`data.Distance`** — расстояние до школы (например, `0.7 mi`, `2.34mi`).
- Количество уникальных записей в колонке очень велико — **297,365 уникальных значений**, то есть почти каждая строка уникальна.

## 🔎 Проблемы
- Данные **неструктурированные**, хранятся в строках.
- Форматы значений неоднородные (`6` vs `6/10`, `0.7 mi` vs `0.7mi`).
- Есть пропуски (`NR` — "No Rating").
- Разные единицы измерения площади (`mi`, иногда без пробела).

## ✅ Выводы
1. Колонка `schools` содержит **важную информацию о ближайших школах**, которая может сильно влиять на стоимость недвижимости.
2. В текущем виде использовать колонку невозможно — данные нужно распарсить и выделить отдельные признаки.
3. Полезные производные признаки:
   - **avg_school_rating** — средний рейтинг школ поблизости.
   - **best_school_rating** — максимальный рейтинг из списка.
   - **num_schools** — количество школ поблизости.
   - **nearest_school_distance** — минимальное расстояние до школы.
   - **school_levels** — наличие школ разных уровней (начальная, средняя, старшая).
4. После выделения этих признаков колонка станет структурированной и удобной для модели.
5. Исходную текстовую колонку `schools` можно удалить после парсинга.

In [21]:
housing_data['sqft'].value_counts()

sqft
0                                          11854
1,200 sqft                                   839
1,000 sqft                                   654
1,100 sqft                                   573
1,800 sqft                                   563
                                           ...  
9,914                                          1
Total interior livable area: 3,055 sqft        1
5,177                                          1
11620                                          1
Total interior livable area: 4,615 sqft        1
Name: count, Length: 25405, dtype: int64

# 📐 Анализ колонки `sqft`

## 📊 Основные значения и частоты
- Пропуски — **40,577** (10.8%).
- Наиболее частые значения:
  - `0` — 11,854 (ошибка или отсутствующее значение).
  - `1,200 sqft` — 839
  - `1,000 sqft` — 654
  - `1,100 sqft` — 573
  - `1,800 sqft` — 563
  - `1,500 sqft` — 547
  - `--` — 535 (явный пропуск).
  - `2,000 sqft` — 523
- Всего уникальных значений: **25,405**.

## 🔎 Общая информация
- Колонка хранит **жилищную площадь** (square feet), но:
  - Данные представлены в виде строк.
  - Формат разный: `1,200 sqft`, `1200`, `--`.
  - Есть некорректные значения (`0`, `--`).

## ✅ Выводы
1. Колонка `sqft` — важный числовой признак (площадь дома), напрямую влияющий на цену.
2. Необходимо очистить данные:
   - Удалить текстовую часть `"sqft"`.
   - Удалить запятые из чисел (`1,200 → 1200`).
   - Превратить колонку в **числовой формат** (`int`).
   - Значения `0` и `--` интерпретировать как **пропуски**.
3. После очистки можно исследовать распределение площади и выявить выбросы.
4. Колонка имеет относительно немного пропусков (10.8%), их можно заполнить медианой или оценить на основе похожих домов.

In [22]:
housing_data['zipcode'].describe()

count     377185
unique      4549
top        32137
freq        2141
Name: zipcode, dtype: object

In [23]:
housing_data['state'].value_counts()

state
FL    115449
TX     83786
NY     24479
CA     23386
NC     21862
TN     18340
WA     13826
OH     12588
IL      8939
NV      8482
GA      6705
CO      6404
PA      5561
MI      5161
DC      4674
AZ      3347
IN      3328
OR      2789
MA      1516
UT      1325
MD      1090
VT       868
MO       866
VA       801
WI       452
NJ       436
ME       259
IA       242
KY        90
OK        49
MS        40
SC        28
MT         7
DE         5
Fl         1
BA         1
AL         1
OT         1
OS         1
Name: count, dtype: int64

In [24]:
housing_data['stories'].value_counts()

stories
1.0                                  67454
2.0                                  55283
1                                    23086
2                                    18146
3.0                                  11275
                                     ...  
1.2                                      1
Manufactured Home, Non-Site Built        1
Bedroom - Split Plan                     1
78                                       1
65.0                                     1
Name: count, Length: 347, dtype: int64

# 🏢 Анализ колонки `stories`

## 📊 Основные значения и частоты
- Пропуски — **150,715** (40.0%).
- Наиболее частые значения:
  - `1.0` — 67,454
  - `2.0` — 55,283
  - `1` — 23,086
  - `2` — 18,146
  - `3.0` — 11,275
  - `0.0` — 7,241 (скорее всего ошибка, так как у дома не может быть 0 этажей).
  - `One` — 5,758
  - `0` — 4,273 (ещё один вариант ошибки).
  - `3` — 4,228
  - `9.0` — 2,918 (выброс, такие дома встречаются редко).
  - `Two` — 2,495
  - `2 Story` — 1,970
  - `1 Story` — 1,253

## 🔎 Общая информация
- Всего уникальных значений: **348**.
- Значения сильно зашумлены:
  - Числовые (`1`, `2`, `3.0`, `2.00`).
  - Текстовые (`One`, `Two`, `2 Story`, `1 Story`).
  - Ошибочные (`0`, `0.0`).
  - Выбросы (`9.0` и выше).
- Большинство домов имеют **1–3 этажа**.

## ✅ Выводы
1. Колонка `stories` — числовой признак (количество этажей), но данные представлены в разных форматах.
2. Необходимо:
   - Преобразовать все значения к числовому виду.
   - Удалить текстовые части (`One` → `1`, `Two` → `2`, `2 Story` → `2`).
   - Заменить некорректные значения (`0`, `0.0`) на `NaN`.
   - Проверить и обработать выбросы (значения > 5 этажей встречаются редко).
3. После очистки колонка станет полезной для модели: количество этажей напрямую влияет на цену.

In [25]:
housing_data['mls-id'].value_counts()

mls-id
No MLS#      3
No           3
1498550      2
39888954     2
608063       2
            ..
1020314      1
A10762436    1
1592770      1
14201834     1
F10202858    1
Name: count, Length: 24907, dtype: int64

In [26]:
housing_data['MlsId'].value_counts()

MlsId
NO MLS                     24
No MLS #                   16
 A, Houston, TX 77008      13
 12A, Orlando, FL 32833    11
 B, Houston, TX 77008       9
                           ..
19092240                    1
RX-10563061                 1
218080001                   1
14154778                    1
10374233                    1
Name: count, Length: 232944, dtype: int64

# 🆔 Анализ колонок `mls-id` и `MlsId`

## 📊 Наблюдения
- **`mls-id`**
  - Пропуски: **93.4%** (352,243).
  - Уникальные значения: много, т.к. это идентификатор.
  - Практически пустая колонка, малополезна.

- **`MlsId`**
  - Пропуски: **17.7%** (66,880).
  - Уникальные значения: много, т.к. это тоже идентификатор.
  - Более полная, чем `mls-id`.

## 🔎 Общая информация
- Оба признака — это **идентификаторы объявлений (MLS = Multiple Listing Service)**.
- Они **не содержат информации о характеристиках недвижимости**.
- Их смысл — уникально идентифицировать запись.
- Для модели предсказания цены они **бесполезны**, потому что идентификатор не влияет на стоимость.

## ✅ Вывод
- **Для анализа и модели предсказания цены колонки `mls-id` и `MlsId` можно удалить.**